# Notebook 02 — Preprocesamiento de Datos

**Proyecto:** Predicción de Churn en Telecomunicaciones  

En este notebook limpiamos, transformamos y preparamos los datos para el entrenamiento de los modelos. Los pasos siguen un orden lógico: limpieza → codificación → escalado → división train/test.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas correctamente.')

Librerías importadas correctamente.


## 2. Carga del dataset

Cargamos el dataset original desde `data/processed/`. Trabajamos sobre una copia para no modificar el dataframe original.

In [2]:
df_raw = pd.read_csv('../data/processed/Telco_customer_churn.csv')
df = df_raw.copy()

print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(3)

Dataset cargado: 7043 filas × 33 columnas


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved


## 3. Eliminación de columnas irrelevantes

Eliminamos columnas que no aportan valor predictivo:
- **Identificadores:** `CustomerID`, `Count`
- **Geográficas:** `Country`, `State`, `City`, `Zip Code`, `Lat Long`, `Latitude`, `Longitude`
- **Variables derivadas o de fuga de datos (data leakage):** `Churn Score`, `CLTV`, `Churn Reason`, `Churn Value` (usaremos `Churn Label` como objetivo)

In [3]:
columnas_eliminar = [
    'CustomerID', 'Count', 'Country', 'State', 'City',
    'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
    'Churn Score', 'CLTV', 'Churn Reason', 'Churn Value'
]

df.drop(columns=columnas_eliminar, inplace=True)
print(f'Columnas restantes: {df.shape[1]}')
print(df.columns.tolist())

Columnas restantes: 20
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label']


## 4. Limpieza de valores nulos en `Total Charges`

`Total Charges` puede contener espacios en blanco cuando el cliente tiene 0 meses de permanencia. Lo convertimos a numérico y **imputamos los nulos con la mediana** para no distorsionar la distribución con la media (que es sensible a valores extremos).

In [4]:
# Convertir a numérico (espacios en blanco → NaN)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

nulos_antes = df['Total Charges'].isnull().sum()
print(f'Valores nulos en Total Charges: {nulos_antes}')

# Imputar con mediana
mediana_tc = df['Total Charges'].median()
df['Total Charges'].fillna(mediana_tc, inplace=True)

print(f'Mediana usada para imputación: ${mediana_tc:.2f}')
print(f'Valores nulos después de imputación: {df["Total Charges"].isnull().sum()}')

Valores nulos en Total Charges: 11
Mediana usada para imputación: $1397.47
Valores nulos después de imputación: 11


## 5. Codificación de variables

Los modelos de Machine Learning requieren que todas las variables sean numéricas. Aplicamos dos estrategias:

### 5.1 Label Encoding — Variables binarias (Yes/No)

Para variables con exactamente dos categorías, **Label Encoding** (0/1) es suficiente y no aumenta la dimensionalidad.

In [5]:
le = LabelEncoder()

# Variables binarias Yes/No
cols_binarias_yn = [
    'Partner', 'Dependents', 'Phone Service', 'Paperless Billing',
    'Online Security', 'Online Backup', 'Device Protection',
    'Tech Support', 'Streaming TV', 'Streaming Movies'
]

# Gender: Male/Female
df['Gender'] = le.fit_transform(df['Gender'])  # Female=0, Male=1

# Senior Citizen: Yes/No
df['Senior Citizen'] = le.fit_transform(df['Senior Citizen'])  # No=0, Yes=1

# Variables Yes/No
for col in cols_binarias_yn:
    df[col] = df[col].map({'No': 0, 'Yes': 1, 'No phone service': 0, 'No internet service': 0})

# Variable objetivo
df['Churn Label'] = df['Churn Label'].map({'No': 0, 'Yes': 1})

print('Label Encoding aplicado a variables binarias.')
print(df[['Gender', 'Senior Citizen', 'Partner', 'Churn Label']].head())

Label Encoding aplicado a variables binarias.
   Gender  Senior Citizen  Partner  Churn Label
0       1               0        0            1
1       0               0        0            1
2       0               0        0            1
3       0               0        1            1
4       1               0        0            1


### 5.2 One-Hot Encoding — Variables multicategoría

Para variables con más de dos categorías, usamos **One-Hot Encoding** para evitar que el modelo interprete un orden ordinal que no existe. `drop_first=True` elimina una categoría redundante para evitar multicolinealidad.

In [6]:
cols_multicat = [
    'Contract', 'Internet Service', 'Payment Method', 'Multiple Lines'
]

df = pd.get_dummies(df, columns=cols_multicat, drop_first=True)

print(f'Dimensiones tras One-Hot Encoding: {df.shape}')
print('Columnas nuevas creadas:')
print([c for c in df.columns if any(m in c for m in cols_multicat)])

Dimensiones tras One-Hot Encoding: (7043, 25)
Columnas nuevas creadas:
['Contract_One year', 'Contract_Two year', 'Internet Service_Fiber optic', 'Internet Service_No', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check', 'Multiple Lines_No phone service', 'Multiple Lines_Yes']


## 6. Separación de features y variable objetivo

In [7]:
X = df.drop(columns=['Churn Label'])
y = df['Churn Label']

print(f'Features (X): {X.shape}')
print(f'Objetivo (y): {y.shape}')
print(f'Distribución de y:\n{y.value_counts()}')

Features (X): (7043, 24)
Objetivo (y): (7043,)
Distribución de y:
Churn Label
0    5174
1    1869
Name: count, dtype: int64


## 7. División train/test (80/20)

Dividimos el dataset en 80% entrenamiento y 20% prueba. `stratify=y` garantiza que ambos subconjuntos mantengan la misma proporción de churn/no-churn. `random_state=42` asegura reproducibilidad.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nProporción churn en train: {y_train.mean()*100:.1f}%')
print(f'Proporción churn en test:  {y_test.mean()*100:.1f}%')

Train: 5634 muestras (80.0%)
Test:  1409 muestras (20.0%)

Proporción churn en train: 26.5%
Proporción churn en test:  26.5%


## 8. Escalado de variables numéricas — StandardScaler

**StandardScaler** transforma cada variable numérica para que tenga **media 0 y desviación estándar 1**. Esto es importante para que variables con rangos diferentes (e.g., `Total Charges` en miles vs. `Tenure Months` en decenas) no dominen artificialmente el modelo.

> **Regla importante:** El scaler se ajusta **solo con datos de entrenamiento** (`fit_transform`) y luego se aplica a los datos de prueba (`transform`). Esto evita *data leakage* (que información del test influya en el entrenamiento).

In [9]:
cols_numericas = ['Tenure Months', 'Monthly Charges', 'Total Charges']

scaler = StandardScaler()

# Ajustar y transformar solo con datos de entrenamiento
X_train[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])

# Aplicar la misma transformación al conjunto de prueba
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])

print('StandardScaler aplicado.')
print('\nEstadísticas de columnas escaladas (train):')
print(X_train[cols_numericas].describe().round(4))

StandardScaler aplicado.

Estadísticas de columnas escaladas (train):
       Tenure Months  Monthly Charges  Total Charges
count      5634.0000        5634.0000      5626.0000
mean         -0.0000          -0.0000         0.0000
std           1.0001           1.0001         1.0001
min          -1.3223          -1.5440        -1.0021
25%          -0.9560          -0.9712        -0.8317
50%          -0.1419           0.1848        -0.3969
75%           0.9165           0.8319         0.6740
max           1.6085           1.7859         2.8005


## 9. Guardado de datasets procesados

Guardamos los cuatro conjuntos resultantes en `data/processed/` para que los notebooks de modelado puedan cargarlos directamente sin repetir el preprocesamiento.

In [10]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Datasets guardados en data/processed/:')
print('  ✓ X_train.csv')
print('  ✓ X_test.csv')
print('  ✓ y_train.csv')
print('  ✓ y_test.csv')

Datasets guardados en data/processed/:
  ✓ X_train.csv
  ✓ X_test.csv
  ✓ y_train.csv
  ✓ y_test.csv


## 10. Resumen del preprocesamiento

| Paso | Acción | Detalle |
|---|---|---|
| **Eliminación** | Columnas irrelevantes | CustomerID, geo, data leakage |
| **Limpieza** | `Total Charges` → numérico | Imputación con mediana |
| **Label Encoding** | Variables binarias | Gender, Senior Citizen, Yes/No |
| **One-Hot Encoding** | Variables multicategoría | Contract, Internet Service, etc. |
| **División** | Train/Test 80/20 | `random_state=42`, `stratify=y` |
| **Escalado** | StandardScaler | Solo en variables numéricas continuas |

Los datos están listos para el modelado. Continúa con el **Notebook 03** (Random Forest).